# 封存原模型对 f696 五输出的逐日身份核对

只读原 Qlib Recorder 的 outer-refit 状态及 Qlib 输入；不拟合、不修改账户，也不读取收益标签。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
import hashlib,json
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
from src.workflow.minute_pit_allocator import _frozen_predictions

root=Path.cwd();card=root/"research/stage1_f696_frozen_model_forward_parity_20261024"
plan_path=card/"plan.json";plan=json.loads(plan_path.read_text());src=plan["source"]
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan["frozen"] and all(v==0 for v in plan["budget"].values())
parent=root/src["parent_card"];f696=root/src["f696_card"]
assert sha(parent/"summary.json")==src["parent_summary_sha256"]
assert sha(parent/"nas_readback.json")==src["parent_readback_sha256"]
assert sha(f696/"summary.json")==src["f696_summary_sha256"]
ps=json.loads((parent/"summary.json").read_text());pr=json.loads((parent/"nas_readback.json").read_text())
fs=json.loads((f696/"summary.json").read_text())
assert ps["nas_evidence"]["sha256"]==src["parent_archive_sha256"] and ps["nas_evidence"]["size_bytes"]==src["parent_archive_size_bytes"]
assert pr["NAS_archive_sha256"]==src["parent_archive_sha256"] and pr["artifact_hashes_checked"]>=118
assert fs["nas_evidence"]["sha256"]==src["f696_archive_sha256"] and fs["nas_evidence"]["size_bytes"]==src["f696_archive_size_bytes"]
pa=parent/"qlib_recorder"/src["parent_experiment_id"]/src["parent_recorder_id"]/"artifacts"
fa=f696/"qlib_recorder"/src["f696_experiment_id"]/src["f696_recorder_id"]/"artifacts"
for name,digest in src["artifact_sha256"].items():assert sha(pa/name)==digest,name
assert sha(fa/"inputs/frame")==src["f696_frame_sha256"]
assert sha(root/src["qlib_provider"]/"qlib_store_manifest.json")==src["provider_manifest_sha256"]
qlib.init(provider_uri=str((root/src["qlib_provider"]).resolve()),region="us",expression_cache=None,dataset_cache=None)
with R.uri_context(str(parent/"qlib_recorder")):
 rec=R.get_recorder(experiment_id=src["parent_experiment_id"],recorder_id=src["parent_recorder_id"])
 source={key:rec.load_object("inputs/"+key) for key in ["frozen_parent_models","features","common","daily_frame","plan"]}
with R.uri_context(str(f696/"qlib_recorder")):
 rec=R.get_recorder(experiment_id=src["f696_experiment_id"],recorder_id=src["f696_recorder_id"])
 frame=rec.load_object("inputs/frame")
calendar=pd.DatetimeIndex(D.calendar(start_time="2011-01-03",end_time="2026-10-09",freq="day"))
assert len(calendar)==3966 and source["features"].index.equals(calendar) and source["common"].index.equals(calendar)
assert source["daily_frame"].index.equals(calendar)
assert len(frame)==944 and pd.DatetimeIndex(frame.signal_date).is_unique
assert len(source["frozen_parent_models"]["outer_refit"]["primary_features"])==39
inputs={"frozen_models":source["frozen_parent_models"],"features":source["features"],"common":source["common"],"frame":source["daily_frame"]}
predicted=_frozen_predictions(inputs,window="outer",plan=source["plan"])
columns=["soxl_weight","soxx_weight","cash_weight","tail_probability","alpha_probability"]
assert list(predicted.columns)==columns
expected=frame.set_index(pd.DatetimeIndex(frame.signal_date)).loc[:,columns].sort_index()
actual=predicted.reindex(expected.index).loc[:,columns]
assert actual.index.equals(expected.index) and actual.notna().all().all()
difference=(actual-expected).abs()
max_abs={key:float(difference[key].max()) for key in columns}
mean_abs={key:float(difference[key].mean()) for key in columns}
pass_gate=all(value<=1e-7 for value in max_abs.values())
result={"status":"original_model_five_output_forward_identity_pass" if pass_gate else "original_model_five_output_forward_identity_failed",
        "plan_sha256":sha(plan_path),"parent_recorder_id":src["parent_recorder_id"],"f696_recorder_id":src["f696_recorder_id"],
        "parent_source_rows":len(source["features"]),"original39_count":39,"recomputed_outer_rows":len(predicted),
        "matched_f696_rows":len(expected),"all_matched_dates_finite":True,
        "max_abs_difference":max_abs,"mean_abs_difference":mean_abs,"identity_gate_pass":pass_gate,
        "new_market_requests":0,"new_model_fits":0,"new_native_accounts":0,"new_bootstraps":0,"new_recorders":0,
        "economic_factor_qualified":False,"independent_validation":False,
        "next_action":"If passed, freeze a separate read-only original39 input-sensitivity audit; if failed, stop and trace the saved active f696 model identity without fitting."}
(card/"result.json").write_text(json.dumps(result,ensure_ascii=False,indent=2)+"\n")
print({"status":result["status"],"rows":len(expected),"max_abs_difference":max_abs})


2026/10/24 16:19:47 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[57393:MainThread](2026-10-24 16:19:48,658) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[57393:MainThread](2026-10-24 16:19:48,659) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[57393:MainThread](2026-10-24 16:19:48,660) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


{'status': 'original_model_five_output_forward_identity_failed', 'rows': 944, 'max_abs_difference': {'soxl_weight': 0.1253344581751272, 'soxx_weight': 0.12063163768821658, 'cash_weight': 0.2359998534321116, 'tail_probability': 0.0, 'alpha_probability': 0.0}}


## 中文结论

只读封存的 39 输入、共同编码输入、outer-refit 原模型状态和 Qlib 日历；重算原模型在 944 个 f696 信号日的五个输出。所有来源 SHA、NAS 回读和日期对齐通过，未读取收益标签、拟合或运行账户。

alpha 与 tail 概率的最大绝对差均为 **0**，但 SOXL、SOXX、现金权重最大差分别为 **0.12533 / 0.12063 / 0.23600**。因此原模型裸前向不是完整 f696 A0 提案；后者还经过某个尚需精确核对的预测到目标处理。此时若用裸模型做特征梯度或遮蔽，并把它解释为 f696 实际配置因果影响，会用错接收对象。

固定身份门失败，决定 `needs_post_model_allocation_target_bridge_no_sensitivity_run`。下一步只读封存的 `inputs/raw_predictions`、`inputs/source_targets` 及原处理代码，验证每步权重变换和五输出身份；通过后才另行冻结模型输入敏感度诊断。不能从本次差异猜测现金修复规则，更没有新的 0bp 候选或 SOXX 成功证据。
